# Semi-Supervised Classification with Graph Convolutional Networks — Implementation / 구현

**Paper**: Kipf, T. N., & Welling, M. (2017). Semi-Supervised Classification with Graph Convolutional Networks. *ICLR 2017*. arXiv:1609.02907

이 노트북은 GCN의 핵심 아이디어를 단계적으로 구현합니다 — renormalized adjacency 계산, 2-layer GCN, semi-supervised node classification, untrained GCN의 표현력 검증.

This notebook walks through the core ideas of GCN — the renormalized adjacency, a 2-layer GCN, semi-supervised node classification, and the surprising representational power of an untrained GCN.

## Outline / 목차
1. **Setup & toy graph** / 설정 및 토이 그래프
2. **Renormalization trick** — $\hat{A} = \tilde{D}^{-1/2}\tilde{A}\tilde{D}^{-1/2}$
3. **GCN layer from scratch** / scratch 구현
4. **Karate club: untrained GCN as feature extractor** / 미학습 GCN의 특징 추출
5. **Karate club: semi-supervised classification (4 labels total)** / 4개 라벨로 준지도 분류
6. **Reproducing Table 3 ablation** — propagation models on a synthetic graph
7. **Summary / 요약**

In [ ]:
import numpy as np
import scipy.sparse as sp
import matplotlib.pyplot as plt
import networkx as nx
import torch
import torch.nn as nn
import torch.nn.functional as F

plt.rcParams['figure.figsize'] = (8, 6)
plt.rcParams['font.size'] = 11
torch.manual_seed(42)
np.random.seed(42)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
print(f'PyTorch: {torch.__version__}')

## Part 1: Toy graph / 토이 그래프

We start with the 4-node cycle graph from the notes' worked example:

```
1 — 2
|   |
3 — 4
```

이 그래프로 인접 행렬, 차수 행렬, renormalized adjacency를 직접 계산해 봅니다.

We compute the adjacency, degree, and renormalized adjacency by hand and verify against the formula.

In [ ]:
A = np.array([
    [0, 1, 1, 0],
    [1, 0, 0, 1],
    [1, 0, 0, 1],
    [0, 1, 1, 0],
], dtype=np.float32)

print('A =')
print(A)
print('\nDegrees:', A.sum(axis=1))

## Part 2: Renormalization trick / 재정규화 트릭

Compute $\hat{A} = \tilde{D}^{-1/2}\tilde{A}\tilde{D}^{-1/2}$ where $\tilde{A} = A + I_N$.

**Why this matters / 왜 중요한가**: 식 (7)의 $I_N + D^{-1/2}AD^{-1/2}$는 고유값 범위 $[0, 2]$를 가져 깊은 네트워크에서 불안정. Self-loop를 추가하고 함께 정규화하면 스펙트럼이 안정화됩니다.

Eq. (7) gives eigenvalues in $[0,2]$, which is unstable when iterated. Adding self-loops *before* normalizing tames the spectrum.

In [ ]:
def renormalize_adjacency(A: np.ndarray) -> np.ndarray:
    """Compute the GCN renormalized adjacency D̃^(-1/2) Ã D̃^(-1/2).

    Args:
        A: Symmetric (N, N) adjacency matrix without self-loops.

    Returns:
        The renormalized adjacency Â of shape (N, N).
    """
    N = A.shape[0]
    A_tilde = A + np.eye(N, dtype=A.dtype)
    d_tilde = A_tilde.sum(axis=1)
    d_inv_sqrt = np.power(d_tilde, -0.5)
    D_inv_sqrt = np.diag(d_inv_sqrt)
    return D_inv_sqrt @ A_tilde @ D_inv_sqrt

A_hat = renormalize_adjacency(A)
print('Â (renormalized) =')
print(np.round(A_hat, 3))
print('\nRow sums (should be ≤ 1):', np.round(A_hat.sum(axis=1), 3))

eigvals_unstable = np.linalg.eigvalsh(np.eye(4) + A / np.sqrt(np.outer(A.sum(1).clip(1), A.sum(1).clip(1))))
eigvals_renorm = np.linalg.eigvalsh(A_hat)
print(f'\nEigenvalues of (I + D^-1/2 A D^-1/2):  {np.round(eigvals_unstable, 3)}  range [{eigvals_unstable.min():.2f}, {eigvals_unstable.max():.2f}]')
print(f'Eigenvalues of renormalized Â:        {np.round(eigvals_renorm, 3)}  range [{eigvals_renorm.min():.2f}, {eigvals_renorm.max():.2f}]')

**관찰 / Observation**: 4-cycle graph에서 모든 노드가 차수 2(이웃) + 1(self) = 3이므로 $\hat{A}_{ij} = 1/3$ for all $\tilde{A}_{ij}=1$. 노드 1의 새 표현은 자기 + 이웃 2 + 이웃 3을 균등 평균. / On the 4-cycle, every node has $\tilde{d}=3$, so non-zero entries of $\hat{A}$ are all $1/3$ — node 1's update is the equal-weight average of itself, node 2, and node 3.

## Part 3: GCN layer from scratch / GCN 레이어 직접 구현

PyTorch로 단일 GCN layer를 구현합니다:

$$H^{(l+1)} = \sigma\!\left(\hat{A} H^{(l)} W^{(l)}\right)$$

Then stack two layers:

$$Z = \mathrm{softmax}\!\left(\hat{A}\,\mathrm{ReLU}(\hat{A} X W^{(0)})\,W^{(1)}\right)$$

In [ ]:
class GCNLayer(nn.Module):
    """Single Graph Convolutional layer (Kipf & Welling, 2017, Eq. 2).

    Implements H^(l+1) = σ(Â H^(l) W^(l)) where Â is precomputed.
    """

    def __init__(self, in_features: int, out_features: int, bias: bool = True):
        super().__init__()
        self.linear = nn.Linear(in_features, out_features, bias=bias)
        nn.init.xavier_uniform_(self.linear.weight)  # Glorot init, as in paper
        if bias:
            nn.init.zeros_(self.linear.bias)

    def forward(self, A_hat: torch.Tensor, H: torch.Tensor) -> torch.Tensor:
        """Forward pass.

        Args:
            A_hat: Renormalized adjacency of shape (N, N).
            H: Node features of shape (N, in_features).

        Returns:
            Aggregated features of shape (N, out_features).
        """
        return A_hat @ self.linear(H)


class GCN(nn.Module):
    """Two-layer GCN for semi-supervised node classification (Eq. 9)."""

    def __init__(self, in_features: int, hidden: int, num_classes: int, dropout: float = 0.5):
        super().__init__()
        self.gc1 = GCNLayer(in_features, hidden)
        self.gc2 = GCNLayer(hidden, num_classes)
        self.dropout = dropout

    def forward(self, A_hat: torch.Tensor, X: torch.Tensor) -> torch.Tensor:
        h = F.relu(self.gc1(A_hat, X))
        h = F.dropout(h, p=self.dropout, training=self.training)
        return self.gc2(A_hat, h)  # logits, softmax applied via cross-entropy

    def embed(self, A_hat: torch.Tensor, X: torch.Tensor) -> torch.Tensor:
        """Return hidden-layer activations (for visualization)."""
        return F.relu(self.gc1(A_hat, X))


model = GCN(in_features=8, hidden=4, num_classes=2)
print(model)
print(f'\nTrainable parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad)}')

## Part 4: Karate club — untrained GCN as feature extractor / 미학습 GCN

Appendix A.1을 재현합니다. **학습 없이도** GCN이 의미 있는 노드 임베딩을 만들어내는지 확인합니다.

Reproduces Appendix A.1: an *untrained* 3-layer GCN with random Glorot weights and identity-matrix input still produces well-separated 2-D embeddings of Zachary's karate club.

In [ ]:
G = nx.karate_club_graph()
N = G.number_of_nodes()
print(f'Karate club: {N} nodes, {G.number_of_edges()} edges')

A_karate = nx.to_numpy_array(G, dtype=np.float32)
A_hat_karate = renormalize_adjacency(A_karate)
A_hat_t = torch.from_numpy(A_hat_karate).float()
X_identity = torch.eye(N)

communities = nx.community.greedy_modularity_communities(G)
node_class = np.zeros(N, dtype=int)
for k, comm in enumerate(communities):
    for v in comm:
        node_class[v] = k
num_classes = len(communities)
print(f'Communities found: {num_classes}, class counts: {np.bincount(node_class)}')

In [ ]:
class UntrainedGCN3(nn.Module):
    """Three-layer GCN with tanh non-linearities (Eq. 13), untrained."""

    def __init__(self, n: int, hidden: int = 4, out: int = 2):
        super().__init__()
        self.W0 = nn.Parameter(torch.empty(n, hidden))
        self.W1 = nn.Parameter(torch.empty(hidden, hidden))
        self.W2 = nn.Parameter(torch.empty(hidden, out))
        for p in [self.W0, self.W1, self.W2]:
            nn.init.xavier_uniform_(p)

    def forward(self, A_hat: torch.Tensor, X: torch.Tensor) -> torch.Tensor:
        h = torch.tanh(A_hat @ X @ self.W0)
        h = torch.tanh(A_hat @ h @ self.W1)
        return torch.tanh(A_hat @ h @ self.W2)


torch.manual_seed(7)
untrained = UntrainedGCN3(N)
with torch.no_grad():
    Z_random = untrained(A_hat_t, X_identity).numpy()

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
pos = nx.spring_layout(G, seed=42)
nx.draw_networkx(G, pos=pos, ax=axes[0], node_color=node_class, with_labels=False,
                 cmap='tab10', node_size=140, edge_color='lightgray')
axes[0].set_title('Karate club network (modularity classes)')
axes[0].axis('off')

for k in range(num_classes):
    mask = node_class == k
    axes[1].scatter(Z_random[mask, 0], Z_random[mask, 1], s=70, label=f'Class {k}')
axes[1].set_title('Untrained 3-layer GCN embedding (random weights)')
axes[1].set_xlabel('dim 1')
axes[1].set_ylabel('dim 2')
axes[1].legend(); axes[1].grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

**관찰 / Observation**: 가중치를 학습하지 않아도 그래프 구조만으로 클래스가 시각적으로 분리되는 경향이 보입니다. 이는 GCN propagation rule이 1-dim Weisfeiler-Lehman의 미분 가능 일반화이기 때문입니다 (Appendix A).

Even with random weights, communities cluster — because the propagation rule is essentially a smoothed WL-1 step (Appendix A).

## Part 5: Semi-supervised classification with 4 labels / 4개 라벨로 준지도 분류

Appendix A.2를 재현. 클래스당 단 한 노드만 라벨링 (총 4 노드 = 11.8%) → 모든 34 노드 분류.

Reproduces Appendix A.2: label only one node per community (4 total ≈ 11.8%) and classify all 34.

In [ ]:
labeled_idx = []
for k in range(num_classes):
    candidates = np.where(node_class == k)[0]
    labeled_idx.append(candidates[0])
labeled_idx = np.array(labeled_idx)
y = torch.from_numpy(node_class).long()
labeled_mask = torch.zeros(N, dtype=torch.bool); labeled_mask[labeled_idx] = True

print(f'Labeled nodes: {labeled_idx.tolist()}  (one per class)')
print(f'Label rate: {labeled_mask.float().mean().item():.3f}')

model = GCN(in_features=N, hidden=8, num_classes=num_classes, dropout=0.0)
optimizer = torch.optim.Adam(model.parameters(), lr=0.01, weight_decay=5e-4)

snapshots = {}
snapshot_iters = {25, 50, 75, 100, 200, 300}
for it in range(1, 301):
    model.train()
    optimizer.zero_grad()
    logits = model(A_hat_t, X_identity)
    loss = F.cross_entropy(logits[labeled_mask], y[labeled_mask])
    loss.backward(); optimizer.step()
    if it in snapshot_iters:
        model.eval()
        with torch.no_grad():
            snapshots[it] = model.embed(A_hat_t, X_identity).numpy()

model.eval()
with torch.no_grad():
    pred = model(A_hat_t, X_identity).argmax(dim=1).numpy()
acc = (pred == node_class).mean()
print(f'\nFinal accuracy on all 34 nodes: {acc:.3f}')

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(13, 8))
for ax, it in zip(axes.flat, sorted(snapshots)):
    Z = snapshots[it]
    if Z.shape[1] > 2:
        Z2d = Z[:, :2]
    else:
        Z2d = Z
    for k in range(num_classes):
        mask = node_class == k
        ax.scatter(Z2d[mask, 0], Z2d[mask, 1], s=60, label=f'C{k}')
    for v in labeled_idx:
        ax.scatter(Z2d[v, 0], Z2d[v, 1], s=180, edgecolors='black', facecolors='none', linewidths=2)
    ax.set_title(f'Iteration {it}')
    ax.grid(True, alpha=0.3)
axes[0, 0].legend(loc='best', fontsize=9)
fig.suptitle('Hidden-layer evolution during training (black rings = labeled nodes)', y=1.02)
plt.tight_layout(); plt.show()

**관찰 / Observation**: 단 4개의 라벨로 학습이 진행되며 4개 커뮤니티가 점점 선형 분리 가능한 형태로 수렴합니다 (논문의 Figure 4 재현). 그래프 구조 자체가 강력한 prior임을 시각적으로 확인할 수 있습니다.

Reproduces Figure 4: with only 4 labeled nodes, the four communities progressively become linearly separable. Visually confirms graph structure as a powerful prior.

## Part 6: Reproducing the propagation-model ablation (Table 3) / 전파 모델 비교 재현

We compare 5 propagation rules from Table 3 on a synthetic stochastic-block-model graph that mimics Cora's structure.

1. **MLP (no graph)**: $X\Theta$
2. **1st-order term only**: $D^{-1/2}AD^{-1/2}X\Theta$
3. **Single parameter (Eq. 7)**: $(I_N + D^{-1/2}AD^{-1/2})X\Theta$
4. **Renormalization (Eq. 8)**: $\tilde{D}^{-1/2}\tilde{A}\tilde{D}^{-1/2}X\Theta$  — **the GCN**
5. **Chebyshev K=2**: $\sum_{k=0}^{2} T_k(\tilde{L})X\Theta_k$

In [ ]:
def make_sbm_graph(sizes=(80, 80, 80), p_in=0.10, p_out=0.005, feat_dim=64, seed=0):
    """Generate a stochastic block model graph with class-correlated features."""
    rng = np.random.default_rng(seed)
    G = nx.stochastic_block_model(sizes, [[p_in if i == j else p_out for j in range(len(sizes))] for i in range(len(sizes))], seed=seed)
    N = G.number_of_nodes()
    A = nx.to_numpy_array(G, dtype=np.float32)
    labels = np.zeros(N, dtype=int)
    offset = 0
    centers = rng.normal(0, 1, size=(len(sizes), feat_dim)).astype(np.float32)
    X = np.zeros((N, feat_dim), dtype=np.float32)
    for k, sz in enumerate(sizes):
        labels[offset:offset+sz] = k
        X[offset:offset+sz] = centers[k] + 0.6 * rng.normal(0, 1, size=(sz, feat_dim)).astype(np.float32)
        offset += sz
    return A, X, labels

A_sbm, X_sbm, y_sbm = make_sbm_graph(seed=1)
N_sbm = A_sbm.shape[0]
K_classes = int(y_sbm.max()) + 1
print(f'SBM graph: N={N_sbm}, edges={int(A_sbm.sum()/2)}, classes={K_classes}, feat_dim={X_sbm.shape[1]}')

rng = np.random.default_rng(0)
label_rate = 0.05
n_labeled = int(label_rate * N_sbm)
perm = rng.permutation(N_sbm)
train_idx = perm[:n_labeled]
val_idx = perm[n_labeled:n_labeled+30]
test_idx = perm[n_labeled+30:]
print(f'Train/val/test sizes: {len(train_idx)}/{len(val_idx)}/{len(test_idx)}  (label rate {label_rate})')

In [ ]:
def sym_norm_no_self(A: np.ndarray) -> np.ndarray:
    """Compute D^(-1/2) A D^(-1/2) without self-loops."""
    d = A.sum(1).clip(min=1e-12)
    d_inv_sqrt = np.power(d, -0.5)
    return (d_inv_sqrt[:, None] * A) * d_inv_sqrt[None, :]

def chebyshev_T(L_tilde: np.ndarray, K: int) -> list:
    """Compute Chebyshev polynomials T_0..T_K of L_tilde, applied as operators."""
    N = L_tilde.shape[0]
    Ts = [np.eye(N, dtype=np.float32), L_tilde.astype(np.float32)]
    for k in range(2, K + 1):
        Ts.append((2 * L_tilde @ Ts[-1] - Ts[-2]).astype(np.float32))
    return Ts

P_renorm = renormalize_adjacency(A_sbm)
P_first = sym_norm_no_self(A_sbm)
P_single = np.eye(N_sbm, dtype=np.float32) + P_first
L_norm = np.eye(N_sbm) - P_first
lam_max = float(np.linalg.eigvalsh(L_norm).max())
L_tilde = (2.0 / lam_max) * L_norm - np.eye(N_sbm)
Cheb_Ts = chebyshev_T(L_tilde, K=2)

print(f'λ_max(L) = {lam_max:.4f}')
print('Operators ready: renorm, first-order, single-param, Chebyshev (K=2).')

In [ ]:
class PropagationModel(nn.Module):
    """Two-layer model with a configurable propagation operator.

    For the Chebyshev variant we accept a list of operators and learn a
    separate weight matrix per polynomial order (as in Defferrard 2016).
    """

    def __init__(self, propagators: list, in_features: int, hidden: int, num_classes: int, dropout: float = 0.5):
        super().__init__()
        self.propagators = propagators  # list of (N, N) tensors
        self.K = len(propagators)
        self.W0 = nn.ParameterList([nn.Parameter(torch.empty(in_features, hidden)) for _ in range(self.K)])
        self.W1 = nn.ParameterList([nn.Parameter(torch.empty(hidden, num_classes)) for _ in range(self.K)])
        for plist in [self.W0, self.W1]:
            for p in plist:
                nn.init.xavier_uniform_(p)
        self.dropout = dropout

    def _layer(self, X: torch.Tensor, Ws: nn.ParameterList) -> torch.Tensor:
        out = 0
        for P, W in zip(self.propagators, Ws):
            out = out + P @ (X @ W)
        return out

    def forward(self, X: torch.Tensor) -> torch.Tensor:
        h = F.relu(self._layer(X, self.W0))
        h = F.dropout(h, p=self.dropout, training=self.training)
        return self._layer(h, self.W1)


def train_propagation(propagators, X_np, y_np, train_idx, test_idx, hidden=32, epochs=200, lr=0.01, wd=5e-4, dropout=0.5, seed=0):
    torch.manual_seed(seed)
    props_t = [torch.from_numpy(P).float() for P in propagators]
    X_t = torch.from_numpy(X_np).float()
    y_t = torch.from_numpy(y_np).long()
    train_t = torch.from_numpy(train_idx).long()
    test_t = torch.from_numpy(test_idx).long()
    K_classes = int(y_np.max()) + 1
    model = PropagationModel(props_t, X_np.shape[1], hidden, K_classes, dropout)
    optim = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=wd)
    for _ in range(epochs):
        model.train(); optim.zero_grad()
        logits = model(X_t)
        loss = F.cross_entropy(logits[train_t], y_t[train_t])
        loss.backward(); optim.step()
    model.eval()
    with torch.no_grad():
        acc = (model(X_t).argmax(1)[test_t] == y_t[test_t]).float().mean().item()
    return acc

configs = {
    'MLP (no graph)': [np.eye(N_sbm, dtype=np.float32)],
    '1st-order only (D^-1/2 A D^-1/2)': [P_first],
    'Single param (I + D^-1/2 A D^-1/2)': [P_single],
    'Renormalization (GCN)': [P_renorm],
    'Chebyshev K=2': Cheb_Ts,
}

results = {}
for name, props in configs.items():
    accs = [train_propagation(props, X_sbm, y_sbm, train_idx, test_idx, seed=s) for s in range(3)]
    results[name] = (float(np.mean(accs)), float(np.std(accs)))
    print(f'{name:42s}  acc = {results[name][0]:.3f} ± {results[name][1]:.3f}')

In [ ]:
names = list(results.keys())
means = [results[n][0] for n in names]
stds = [results[n][1] for n in names]
colors = ['gray', 'steelblue', 'cornflowerblue', 'crimson', 'mediumseagreen']

fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(range(len(names)), means, yerr=stds, color=colors, alpha=0.85, capsize=4)
ax.set_xticks(range(len(names))); ax.set_xticklabels(names, rotation=20, ha='right')
ax.set_ylabel('Test accuracy')
ax.set_ylim(0, 1.05)
ax.set_title('Propagation-model ablation on synthetic SBM (≈ Table 3)')
ax.grid(True, axis='y', alpha=0.3)
for i, (m, s) in enumerate(zip(means, stds)):
    ax.text(i, m + s + 0.02, f'{m:.2f}', ha='center', fontsize=10)
plt.tight_layout(); plt.show()

**관찰 / Observation**: 결과의 정성적 패턴이 논문의 Table 3과 일치합니다 — MLP만 사용 시 정확도가 가장 낮고, renormalization variant(GCN)가 비슷한 매개변수 수의 다른 그래프 변형보다 우수합니다. Chebyshev $K=2$는 더 많은 매개변수로 비슷한 성능. / The qualitative pattern matches Table 3: MLP is worst, renormalized GCN is best, Chebyshev K=2 is comparable but uses more parameters per layer.

## Part 7: Summary / 요약

| Concept / 개념 | This paper / 이 논문 | Modern equivalent / 현대 동등물 |
|---|---|---|
| Spectral graph convolution | Eq. (3): $g_\theta \star x = U g_\theta(\Lambda) U^\top x$ | 초기 spectral methods (Bruna 2014, ChebNet 2016) |
| Localization | Chebyshev $K$-truncation (Eq. 5) | Spatial neighborhood aggregation (GraphSAGE) |
| Renormalization trick | $\tilde{D}^{-1/2}\tilde{A}\tilde{D}^{-1/2}$ (Eq. 8) | Standard normalization in PyG / DGL |
| Layer-wise propagation | $\hat{A} H^{(l)} W^{(l)}$ (Eq. 2) | Message-passing (MPNN), GraphSAGE, GAT |
| Semi-supervised SSL | Drop Laplacian regularizer; condition on $A$ | All modern transductive GNNs |
| WL connection | Appendix A: GCN ≈ differentiable WL-1 | GIN (Xu et al. 2019), expressivity theory |
| Untrained features | Random GCN gives meaningful embeddings | Implicit-prior GNNs, simplified GCN (SGC) |

### Takeaway / 결론

**한국어**: GCN의 천재성은 *단순함*에 있습니다. 한 줄의 propagation rule, self-loop 추가, 정규화. 이 세 가지 변경이 spectral 이론을 실용적이고 확장 가능한 형태로 변환했고, 이후 거의 모든 GNN의 출발점이 되었습니다.

**English**: GCN's genius is its *simplicity*. A one-line propagation rule, a self-loop, and a normalization — three changes that turned spectral graph theory into a practical, scalable model and seeded almost every GNN that followed.